# T8.6 — CafeBERT đóng băng thay PhoBERT, bài category

Chỉ đổi encoder: `uitnlp/CafeBERT` (XLM-R-large, 1024 chiều, SentencePiece trên văn bản
**chưa tách từ**). Split, head, hyper-param, `max_len=256` giữ nguyên để so công bằng.

Mốc để so (dev, PhoBERT đóng băng): `dl-cat-ce-cpu-0920` (dense) macro-F1 **0,6030**,
`dl-cat-rnn-ce` (rnn) **0,6046**.

**Trước khi chạy:** Runtime → Change runtime type → **GPU** (A100/L4 nếu có; T4 chạy được nhưng chậm — mô hình large).
Bước này chỉ chạy **baseline dense**; head rnn (cần cache token ≈ 18 GB) để sau.

In [ ]:
!nvidia-smi | head -15
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('vietjobs'):
    !git clone -q -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
!git log --oneline -1
# KHÔNG cài lại torch — giữ bản CUDA sẵn có của Colab (requirements-dl.txt nói rõ).
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" sentencepiece protobuf pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

## Khôi phục splits (bỏ bước encode PhoBERT)

In [ ]:
!python notebooks/colab_setup.py --family raw --no-train-token --skip-tests

## Encode CafeBERT — chỉ pooled (train+dev)

Head dense chỉ đọc vector đã gộp `[n, 1024]`, không cần cache token 18 GB. Sidecar `.json` ghi `truncated_frac`: tỷ lệ tin bị cắt ở 256 token SentencePiece.

In [ ]:
!python -m vietjobs.dl.encode --task category --splits train dev --encoder cafebert --device cuda --batch 64
!cat artifacts/embeddings/cafebert-dev-raw-len256.json

In [ ]:
# Sao lưu cache pooled nhỏ lên Drive (vài trăm MB) để khỏi encode lại.
!mkdir -p /content/drive/MyDrive/vietjobs_cache
!cp artifacts/embeddings/cafebert-*-len256.npy artifacts/embeddings/cafebert-*-len256*.json /content/drive/MyDrive/vietjobs_cache/

In [ ]:
!python -m pytest -q tests/test_dl_encode.py tests/test_dl_heads.py

## Hàm lưu kết quả — in F1 score, zip, lưu Drive, tải về

## Huấn luyện — cùng cấu hình với hai mốc PhoBERT

In [ ]:
# Head dense trên cpu, như dl-cat-ce-cpu-0920 (Rule 13 mục 5).
!python -m vietjobs.dl.train_dl --task category --encoder cafebert --head dense --device cpu --run-id dl-cat-cafe-dense
save_run("dl-cat-cafe-dense")

In [ ]:
# Thanh tham chiếu: linear probe trên cùng vector CafeBERT.
!python scripts/probe_embeddings.py --task category --encoder cafebert --standardize

## Mang về máy chính

Tải `dl-cat-cafe-dense.zip` (cũng ở `MyDrive/vietjobs_runs/`), cùng
`docs/04-results.md` của phiên này (có dòng probe). Giải nén vào `artifacts/<run_id>/` rồi đưa lại
để so bootstrap với `dl-cat-ce-cpu-0920` / `dl-cat-rnn-ce` và cập nhật tài liệu.